# 🚀 Colab AI Studio — Worker Distant GPU (Tesla T4)

Ce notebook transforme cette session Google Colab en worker de calcul distant pour **Colab AI Studio**.

**Prérequis** : Assurez-vous d'avoir activé l'accélération GPU : `Exécution > Modifier le type d'exécution > T4 GPU`.

In [ ]:
# [1/4] Vérification du GPU NVIDIA Tesla T4
!nvidia-smi

In [ ]:
# [2/4] Installation des dépendances du worker et du tunnel
!pip install -q fastapi uvicorn pydantic pycloudflared pyngrok nest_asyncio httpx

In [ ]:
# [3/4] Clonage du dépôt et initialisation de l'automate ModelSwapper
!git clone https://github.com/personnal-agent/colab-ai-studio.git /content/colab-ai-studio \
  2>/dev/null || (cd /content/colab-ai-studio && git pull)

import sys

if '/content/colab-ai-studio' not in sys.path:
    sys.path.insert(0, '/content/colab-ai-studio')

from app.colab_worker.model_swapper import ModelSwapper
from app.colab_worker.server import create_worker_app

swapper = ModelSwapper(idle_timeout_seconds=300.0)
app = create_worker_app(model_swapper=swapper)

In [ ]:
# [4/4] Ouverture du tunnel Cloudflare et lancement du serveur uvicorn
import nest_asyncio
import uvicorn
from pycloudflared import try_cloudflare

nest_asyncio.apply()

PORT = 8000
tunnel = try_cloudflare(port=PORT)
tunnel_url = tunnel.tunnel.rstrip('/')

print("\n======================================================")
print(f"🚀 COLAB WORKER ACTIF SUR : {tunnel_url}")
print("Copiez cette URL dans Colab AI Studio (Statut Colab / Tunnel URL)")
print("======================================================\n")

uvicorn.run(app, host="127.0.0.1", port=PORT)